In [1]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "cryptohftdata"])
subprocess.run([sys.executable, "-m", "pip", "install", "polars"])

  Attempting uninstall: coverage
    Found existing installation: coverage 7.9.2
    Uninstalling coverage-7.9.2:
      Successfully uninstalled coverage-7.9.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cryptohftdata]



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 13.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 85.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


CompletedProcess(args=['/usr/bin/python3.12', '-m', 'pip', 'install', 'polars'], returncode=0)

## Download all Files (Multi-Core)

In [7]:
import pandas as pd
import cryptohftdata as chd
from datetime import datetime, timedelta
import os
import gc
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- CONFIGURATION ---
API_KEY = "467ab24e39287d1280d30cf4b3cd803217ac2b73bac1e71201251117c7cad952"
START_DATE = "2025-10-01"
END_DATE = "2025-10-05"
SYMBOL = "BTCUSDT"
EXCHANGE = chd.exchanges.BINANCE_FUTURES
MAX_WORKERS = 6  # <- tune this (start low!)

DROP_HEADERS = [
    'some_unused_column', 'symbol',
    'transaction_time', 'event_time', 'timestamp', 'prev_final_update_id'
]


def optimize_floats(df):
    floats = df.select_dtypes(include=['float64']).columns
    df[floats] = df[floats].astype('float32')
    return df


def get_date_list(start, end):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    delta = end_dt - start_dt
    return [(start_dt + timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(delta.days + 1)]


def process_single_date(date_str, category):
    """Runs inside each process."""
    client = chd.CryptoHFTDataClient(api_key=API_KEY)

    cat_name, fetch_method_name = category
    fetch_func = getattr(client, fetch_method_name)

    os.makedirs('orderbook_raw', exist_ok=True)
    file_path = f"orderbook_raw/{date_str}_{SYMBOL}.parquet"

    if os.path.exists(file_path):
        return f"[SKIP] {date_str}"

    try:
        df = fetch_func(
            symbol=SYMBOL,
            exchange=EXCHANGE,
            start_date=date_str,
            end_date=date_str
        )

        if df is None or df.empty:
            return f"[EMPTY] {date_str}"

        # Drop unused columns
        df.drop(columns=[c for c in DROP_HEADERS if c in df.columns],
                errors='ignore', inplace=True)

        # Convert time
        if 'received_time' in df.columns:
            df['received_time'] = pd.to_datetime(df['received_time'], unit='ns')

        # Optimize
        df = optimize_floats(df)

        # Save
        df.to_parquet(
            file_path,
            engine='pyarrow',
            compression='zstd',
            compression_level=9,
            index=False
        )

        rows = len(df)

        del df
        gc.collect()

        return f"[SAVED] {date_str} | Rows: {rows:,}"

    except Exception as e:
        if 'df' in locals():
            del df
        gc.collect()
        return f"[ERROR] {date_str}: {e}"


def download_and_store():
    dates = get_date_list(START_DATE, END_DATE)

    categories = [
        ("orderbook", "get_orderbook"),
        #("trades", "get_trades"),
        #("open_interest", "get_open_interest"),
    ]

    for category in categories:
        cat_name, _ = category
        print(f"\n>>> Starting Category: {cat_name.upper()}")

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(process_single_date, date, category)
                for date in dates
            ]

            for future in as_completed(futures):
                print(" ", future.result())


if __name__ == "__main__":
    download_and_store()
    print("\nDownload process complete.")


>>> Starting Category: ORDERBOOK


Error downloading binance_futures/2025-10-04/07/BTCUSDT_orderbook.parquet.zst: HTTPSConnectionPool(host='api.cryptohftdata.com', port=443): Read timed out. (read timeout=30)


  [SAVED] 2025-10-05 | Rows: 96,675,608
  [SAVED] 2025-10-01 | Rows: 108,559,553
  [SAVED] 2025-10-04 | Rows: 70,137,356
  [SAVED] 2025-10-02 | Rows: 112,171,675
  [SAVED] 2025-10-03 | Rows: 117,993,659

Download process complete.


## Now Split files into multiple segments:

In [9]:
import polars as pl
from tqdm import tqdm
import os

def split_parquet_by_interval(input_file, num_intervals, prefix):
    # 1. Scan the file lazily
    lf = pl.scan_parquet(input_file)

    # 2. Get the start and end times
    # Since it's already datetime[ns], we just pull the min/max
    print("Analyzing file metadata...")
    time_bounds = lf.select([
        pl.col("received_time").min().alias("start"),
        pl.col("received_time").max().alias("end")
    ]).collect()

    start_time = time_bounds["start"][0]
    end_time = time_bounds["end"][0]

    if start_time is None or end_time is None:
        print("Error: Could not find time bounds. Is the 'received_time' column empty?")
        return

    total_duration = end_time - start_time
    interval_duration = total_duration / num_intervals

    print(f"Start: {start_time}")
    print(f"End:   {end_time}")
    print(f"Splitting into {num_intervals} chunks of ~{interval_duration} each.\n")

    # 3. Create output directory
    output_dir = "orderbook_chunks"
    os.makedirs(output_dir, exist_ok=True)

    # 4. Filter and save in a memory-efficient loop
    for i in tqdm(range(num_intervals), desc="Processing Chunks"):
        chunk_start = start_time + (i * interval_duration)
        chunk_end = start_time + ((i + 1) * interval_duration)

        # We use collect() inside the loop so only ONE slice is in RAM at a time
        chunk_df = (
            lf.filter(
                (pl.col("received_time") >= chunk_start) & 
                (pl.col("received_time") < chunk_end)
            )
            .collect()
        )

        if not chunk_df.is_empty():
            output_filename = f"{output_dir}/{prefix}_chunk_{i+1:02d}.parquet"
            chunk_df.write_parquet(output_filename)
        
        # Explicitly clear chunk from memory (optional, but good for very tight RAM)
        del chunk_df

    print(f"\nDone! Files are in the '{output_dir}' folder.")

import os

# Assuming split_parquet_by_interval is defined above...

if __name__ == "__main__":
    # Specify the folder containing your parquet files
    folder_path = "./orderbook_raw" 
    
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' not found.")
    else:
        # Get a list of all parquet files in the directory
        files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]
        
        if not files:
            print("No .parquet files found in the directory.")
        else:
            try:
                #intervals = int(input("Enter the number of intervals (e.g., 24 for hourly): "))
                intervals = 6
                
                for filename in files:
                    # Construct the full path to the file
                    file_path = os.path.join(folder_path, filename)
                    
                    print(f"--- Processing: {filename} ---")
                    split_parquet_by_interval(file_path, intervals, filename)
                
                print("\nAll files processed successfully.")
                
            except ValueError:
                print("Please enter a valid whole number.")

--- Processing: 2025-10-01_BTCUSDT.parquet ---
Analyzing file metadata...
Start: 2025-10-01 00:00:00.034583
End:   2025-10-01 23:59:59.960229
Splitting into 6 chunks of ~3:59:59.987608 each.



Processing Chunks: 100%|██████████| 6/6 [00:01<00:00,  4.12it/s]



Done! Files are in the 'orderbook_chunks' folder.
--- Processing: 2025-10-02_BTCUSDT.parquet ---
Analyzing file metadata...
Start: 2025-10-02 00:00:00.017453
End:   2025-10-02 23:59:59.976674
Splitting into 6 chunks of ~3:59:59.993204 each.



Processing Chunks: 100%|██████████| 6/6 [00:01<00:00,  4.09it/s]



Done! Files are in the 'orderbook_chunks' folder.
--- Processing: 2025-10-05_BTCUSDT.parquet ---
Analyzing file metadata...
Start: 2025-10-05 00:00:00.008585
End:   2025-10-05 23:59:59.949165
Splitting into 6 chunks of ~3:59:59.990097 each.



...


Done! Files are in the 'orderbook_chunks' folder.
--- Processing: 2025-10-04_BTCUSDT.parquet ---
Analyzing file metadata...
Start: 2025-10-04 00:00:00.061273
End:   2025-10-04 23:59:59.956019
Splitting into 6 chunks of ~3:59:59.982458 each.



Processing Chunks: 100%|██████████| 6/6 [00:00<00:00,  6.45it/s]



Done! Files are in the 'orderbook_chunks' folder.
--- Processing: 2025-10-03_BTCUSDT.parquet ---
Analyzing file metadata...
Start: 2025-10-03 00:00:00.032090
End:   2025-10-03 23:59:59.976217
Splitting into 6 chunks of ~3:59:59.990688 each.



Processing Chunks: 100%|██████████| 6/6 [00:01<00:00,  3.94it/s]


Done! Files are in the 'orderbook_chunks' folder.

All files processed successfully.


## Now reconstuct it?

## Reconstruction by Gemini:

In [12]:
import os
import glob
import pandas as pd
import numpy as np
import json
from numba import njit
from datetime import datetime

# Configuration
INPUT_FOLDER = "orderbook_chunks"
OUTPUT_FOLDER = "features"
CHECKPOINT_FOLDER = "checkpoints"
OUTPUT_FILE = "orderbook_features.parquet"
INTERVAL_NS = 30 * 1_000_000_000  # 30 seconds
LEVELS_TO_EXTRACT = 10
SAVE_INTERVAL_FILES = 15  # Save every 15 subfiles

@njit
def compute_features_numba(bid_prices, bid_qtys, ask_prices, ask_qtys, timestamp):
    """
    Optimized feature computation using Numba.
    Expects sorted arrays (bids descending, asks ascending).
    """
    if len(bid_prices) == 0 or len(ask_prices) == 0:
        return np.zeros(20) # Return empty feature vector

    best_bid = bid_prices[0]
    best_bid_qty = bid_qtys[0]
    best_ask = ask_prices[0]
    best_ask_qty = ask_qtys[0]
    
    mid_price = (best_bid + best_ask) / 2.0
    spread = best_ask - best_bid
    spread_bps = (spread / mid_price) * 10000.0 if mid_price > 0 else 0.0
    micro_price = (best_bid * best_ask_qty + best_ask * best_bid_qty) / (best_bid_qty + best_ask_qty)

    # Output indices: 
    # 0: ts, 1: bb, 2: ba, 3: mid, 4: spread, 5: bps, 6: micro
    # 7-9: bid_vol (1,5,10), 10-12: ask_vol, 13-15: obi, 16: b_dens, 17: a_dens
    res = np.zeros(20)
    res[0] = timestamp
    res[1] = best_bid
    res[2] = best_ask
    res[3] = mid_price
    res[4] = spread
    res[5] = spread_bps
    res[6] = micro_price

    # Aggregated Volumes and OBI
    ns = np.array([1, 5, 10])
    for i in range(3):
        n = ns[i]
        b_vol = np.sum(bid_qtys[:n])
        a_vol = np.sum(ask_qtys[:n])
        res[7+i] = b_vol
        res[10+i] = a_vol
        
        total_v = b_vol + a_vol
        res[13+i] = (b_vol - a_vol) / total_v if total_v > 0 else 0.0

    # Densities
    bid_range = bid_prices[0] - bid_prices[min(len(bid_prices)-1, 9)]
    ask_range = ask_prices[min(len(ask_prices)-1, 9)] - ask_prices[0]
    
    res[16] = res[9] / bid_range if bid_range > 0 else 0.0
    res[17] = res[12] / ask_range if ask_range > 0 else 0.0

    return res

def get_sorted_book(book_dict, reverse=False):
    """Utility to convert dict to sorted numpy arrays."""
    if not book_dict:
        return np.array([]), np.array([])
    prices = np.sort(np.array(list(book_dict.keys())))
    if reverse:
        prices = prices[::-1]
    
    qtys = np.array([book_dict[p] for p in prices])
    return prices.astype(np.float64), qtys.astype(np.float64)

def load_progress():
    path = os.path.join(CHECKPOINT_FOLDER, "metadata.json")
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {"processed_files": [], "next_snapshot_time": None}

def save_progress(processed_files, next_snapshot_time, features_acc):
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)
    
    # Cast next_snapshot_time to standard Python int to avoid JSON serialization error
    serializable_time = int(next_snapshot_time) if next_snapshot_time is not None else None
    
    # Save Metadata
    with open(os.path.join(CHECKPOINT_FOLDER, "metadata.json"), 'w') as f:
        json.dump({"processed_files": processed_files, "next_snapshot_time": serializable_time}, f)
    
    # Save partial features
    if features_acc:
        cols = [
            'timestamp', 'best_bid', 'best_ask', 'mid_price', 'spread', 'spread_bps', 'micro_price',
            'bid_vol_1', 'bid_vol_5', 'bid_vol_10', 'ask_vol_1', 'ask_vol_5', 'ask_vol_10',
            'obi_1', 'obi_5', 'obi_10', 'bid_density', 'ask_density', 'empty1', 'empty2'
        ]
        # Convert list of numpy arrays to a DataFrame
        df = pd.DataFrame(np.array(features_acc), columns=cols)
        df.to_parquet(os.path.join(CHECKPOINT_FOLDER, f"partial_{len(processed_files)}.parquet"))

def process_orderbook_stream():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(CHECKPOINT_FOLDER, exist_ok=True)
    
    progress = load_progress()
    processed_set = set(progress["processed_files"])
    next_snapshot_time = progress["next_snapshot_time"]

    bids = {} 
    asks = {}
    
    file_pattern = os.path.join(INPUT_FOLDER, "*.parquet")
    files = sorted(glob.glob(file_pattern))
    
    if not files:
        print("No input files found.")
        return

    all_features = []
    file_counter = 0
    
    print(f"Starting processing. Total files: {len(files)}")

    for file_path in files:
        if file_path in processed_set:
            continue

        print(f"[{datetime.now().strftime('%H:%M:%S')}] Processing: {os.path.basename(file_path)}")
        df = pd.read_parquet(file_path)

        if not pd.api.types.is_integer_dtype(df['received_time']):
            df['received_time'] = df['received_time'].values.astype('int64')

        times = df['received_time'].values
        sides = df['side'].values
        prices = df['price'].values
        qtys = df['quantity'].values

        for i in range(len(times)):
            ts = times[i]
            
            if next_snapshot_time is None:
                next_snapshot_time = ts + INTERVAL_NS

            while ts >= next_snapshot_time:
                b_p, b_q = get_sorted_book(bids, reverse=True)
                a_p, a_q = get_sorted_book(asks, reverse=False)
                
                feat_vec = compute_features_numba(b_p, b_q, a_p, a_q, next_snapshot_time)
                if feat_vec[1] > 0: 
                    all_features.append(feat_vec)
                
                next_snapshot_time += INTERVAL_NS

            p_val = float(prices[i])
            q_val = float(qtys[i])
            target = bids if sides[i] == 'bid' else asks
            
            if q_val == 0:
                target.pop(p_val, None)
            else:
                target[p_val] = q_val

        file_counter += 1
        progress["processed_files"].append(file_path)

        if file_counter % SAVE_INTERVAL_FILES == 0:
            print(f"--- Checkpoint reached at {file_counter} files ---")
            save_progress(progress["processed_files"], next_snapshot_time, all_features)

    if not all_features:
        print("No features generated.")
        return

    cols = [
        'timestamp', 'best_bid', 'best_ask', 'mid_price', 'spread', 'spread_bps', 'micro_price',
        'bid_vol_1', 'bid_vol_5', 'bid_vol_10', 'ask_vol_1', 'ask_vol_5', 'ask_vol_10',
        'obi_1', 'obi_5', 'obi_10', 'bid_density', 'ask_density', 'empty1', 'empty2'
    ]
    
    final_df = pd.DataFrame(np.array(all_features), columns=cols)
    final_df = final_df.drop(columns=['empty1', 'empty2'])

    print("Computing target variables...")
    PERIODS_15_MIN = int((15 * 60 * 1_000_000_000) / INTERVAL_NS)
    final_df['future_mid_price_15m'] = final_df['mid_price'].shift(-PERIODS_15_MIN)
    final_df['target_price_up'] = (final_df['future_mid_price_15m'] > final_df['mid_price']).astype(int)
    final_df = final_df.dropna(subset=['future_mid_price_15m'])

    output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILE)
    final_df.to_parquet(output_path, index=False)
    print(f"Complete! Saved to {output_path}")

if __name__ == "__main__":
    process_orderbook_stream()

Starting processing. Total files: 30
[16:40:44] Processing: 2025-10-01_BTCUSDT.parquet_chunk_01.parquet
[16:40:53] Processing: 2025-10-01_BTCUSDT.parquet_chunk_02.parquet
[16:41:03] Processing: 2025-10-01_BTCUSDT.parquet_chunk_03.parquet
[16:41:17] Processing: 2025-10-01_BTCUSDT.parquet_chunk_04.parquet
[16:41:32] Processing: 2025-10-01_BTCUSDT.parquet_chunk_05.parquet
[16:41:46] Processing: 2025-10-01_BTCUSDT.parquet_chunk_06.parquet
[16:41:58] Processing: 2025-10-02_BTCUSDT.parquet_chunk_01.parquet
[16:42:13] Processing: 2025-10-02_BTCUSDT.parquet_chunk_02.parquet
[16:42:25] Processing: 2025-10-02_BTCUSDT.parquet_chunk_03.parquet
[16:42:37] Processing: 2025-10-02_BTCUSDT.parquet_chunk_04.parquet
[16:42:54] Processing: 2025-10-02_BTCUSDT.parquet_chunk_05.parquet
[16:43:11] Processing: 2025-10-02_BTCUSDT.parquet_chunk_06.parquet
[16:43:24] Processing: 2025-10-03_BTCUSDT.parquet_chunk_01.parquet
[16:43:38] Processing: 2025-10-03_BTCUSDT.parquet_chunk_02.parquet
[16:43:50] Processing: 20

TypeError: Object of type int64 is not JSON serializable